# Registered Motorized Vehicle Fleet Observations Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Registered Motorized Vehicle Fleet dataset for Ho Chi Minh City, Viet Nam, 2024–2026 using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Install mlcroissant (if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.py4b-vtrr/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")# Display identifier, license, and keywords
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
List all available record sets, fields, and their `@id` identifiers.

In [ ]:
# Retrieve all record sets using the Croissant metadata
record_set_ids = []
field_map = {}
column_map = {}

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for record_set in metadata.recordSet:
        rs_id = record_set['@id'] if isinstance(record_set, dict) and '@id' in record_set else record_set
        record_set_ids.append(rs_id)
        print(f"RecordSet @id: {rs_id}")
        # List fields for each record set
        if isinstance(record_set, dict) and 'field' in record_set:
            for field in record_set['field']:
                field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
                field_map.setdefault(rs_id, []).append(field_id)
                print(f"  Field @id: {field_id}")
            # List columns if present
        if isinstance(record_set, dict) and 'column' in record_set:
            for column in record_set['column']:
                col_id = column['@id'] if isinstance(column, dict) and '@id' in column else column
                column_map.setdefault(rs_id, []).append(col_id)
                print(f"  Column @id: {col_id}")

# If no record sets found, try to enumerate from raw schema
if not record_set_ids:
    # Try to retrieve record sets from dataset.metadata.to_json()
    meta_json = dataset.metadata.to_json()
    if 'recordSet' in meta_json and meta_json['recordSet']:
        for record_set in meta_json['recordSet']:
            rs_id = record_set.get('@id', None)
            if rs_id:
                record_set_ids.append(rs_id)
                print(f"RecordSet @id: {rs_id}")
                # List fields for each record set
                if 'field' in record_set:
                    for field in record_set['field']:
                        field_id = field.get('@id', None)
                        if field_id:
                            field_map.setdefault(rs_id, []).append(field_id)
                            print(f"  Field @id: {field_id}")
                # List columns if present
                if 'column' in record_set:
                    for column in record_set['column']:
                        col_id = column.get('@id', None)
                        if col_id:
                            column_map.setdefault(rs_id, []).append(col_id)
                            print(f"  Column @id: {col_id}")

# Show found record sets, fields and columns
print("\nRecordSet IDs:", record_set_ids)
print("\nField Map:", field_map)
print("\nColumn Map:", column_map)

## 3. Data Extraction
Load records from the selected record set(s) into a DataFrame for analysis. Refer to the `@id` values from the data overview above.

In [ ]:
# Choose record sets to extract (update with real @id from dataset)
record_sets = record_set_ids  # Use all found; update for fewer if desired

dataframes = {}
for record_set_id in record_sets:
    try:
        # Load all records from the record set
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet '{record_set_id}' loaded with shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load RecordSet '{record_set_id}': {e}")

## 4. Exploratory Data Analysis (EDA)
Process the data: filter records based on criteria, normalize numeric fields, and group by attributes.

For example, filter vehicles with observed count above threshold, normalize values, and group by vehicle type or spatial unit.

**Note:** Replace `<record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` with actual `@id` values from your dataset. This notebook demonstrates on the first loaded record set if possible.

In [ ]:
# Example: process the first record set
if len(record_sets) > 0:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    print(f"Analyzing RecordSet: {record_set_id}")

    # Guess numeric and group fields:
    numeric_field = None
    group_field = None
    # Search for likely numeric fields
    for col in df.columns:
        if col.lower() in ['vehicle_count', 'value', 'registered_count', 'fleet_count']:
            numeric_field = col
            break
    # Group fields commonly spatial or type
    for col in df.columns:
        if col.lower() in ['vehicle_type', 'class', 'perimeter', 'spatial_unit', 'epistemic_status']:
            group_field = col
            break
    # Default fallback
    if not numeric_field:
        # Try to find numeric columns by dtype
        num_cols = df.select_dtypes('number').columns
        if len(num_cols) > 0:
            numeric_field = num_cols[0]
    if not group_field:
        group_field = df.columns[0]
    print("Numeric field:", numeric_field)
    print("Group field:", group_field)

    # Filter records
    threshold = 10000
    if numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized values for {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by group_field if present
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print(f"No numeric field found for EDA in record set '{record_set_id}'.")

## 5. Visualization
Visualize the distribution and relationships of key fields, such as vehicle counts across types or spatial units.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets) > 0 and numeric_field and group_field:
    plt.figure(figsize=(10, 6))
    try:
        sns.barplot(
            data=filtered_df,
            x=group_field,
            y=numeric_field,
            ci=None
        )
        plt.title(f"Mean {numeric_field} by {group_field} (Filtered > {threshold})")
        plt.xticks(rotation=30, ha='right')
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Visualization failed: {e}")

    # Plot normalized values
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, bins=10)
        plt.title(f"Normalized Distribution for {numeric_field} (Filtered > {threshold})")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.tight_layout()
        plt.show()
else:
    print("Not enough fields for visualization.")

## 6. Conclusion
This notebook used the `mlcroissant` library to load, explore, and visualize the FAIR² Registered Motorized Vehicle Fleet dataset for Ho Chi Minh City, Viet Nam, 2024–2026.

Key findings include:
- Transparent access to reconstructed vehicle fleet figures across administrative reforms.
- Capability to filter, normalize, and group fleet data by key attributes such as vehicle type or spatial unit.
- Visual summaries highlight fleet composition and enable deeper emission modeling or provenance analysis.

Further work could involve advanced analysis, temporal change studies, or integrating additional environmental data.